In [ ]:
# Energy features
# Include only melting temperature, GC content and free energy change for guide and target 

from nupack import *
import RNA
import math
import itertools
import numpy as np
from Bio.SeqUtils import MeltingTemp as mt
from Bio.Seq import Seq
import collections

# Record positions of base pairs with the order of base positions in "left_paren" corresponding to the base positions in "right_paren"
def find_pair(structure):
    left_paren = []
    right_paren = []
    left_paren_pre = []

    for i in range (0, len(structure)):
        if structure[i] == '(':
            left_paren_pre.append(i)
        if structure[i] == ')':
            right_paren.append(i)
            left_paren.append(left_paren_pre[-1])
            left_paren_pre = left_paren_pre[:-1]

    return left_paren, right_paren

def pairing_state_correction(structure, left_paren, right_paren, start_position, end_position):
    structure_list = list(structure)
    for i in range (0, len(left_paren)):
        if left_paren[i] >= start_position and left_paren[i] <= end_position:
            structure_list[left_paren[i]] = '.'
            structure_list[right_paren[i]] = '.'
        if right_paren[i] >= start_position and right_paren[i] <= end_position:
            structure_list[left_paren[i]] = '.'
            structure_list[right_paren[i]] = '.'
        structure = ''.join(structure_list)
    return structure

def find_substring_positions(substring, long_string):

    positions = []
    start = 0
    while True:
        index = long_string.find(substring, start)
        if index == -1:  # No more occurrences found.
            break
        positions.append(index)
        start = index + 1  # Move past the last found index.
    return positions

def DNA_to_RNA(DNA):
    match = {'A': 'A', 'C': 'C', 'G': 'G', 'T': 'U'}
    return ''.join(match.get(base, base) for base in (DNA))

def normalize_guide(value):
    normalized_value = (value - (-7)) / (0 - (-7))
    return normalized_value

def normalize_target_bh(value):
    normalized_value = (value - (-39)) / (0 - (-39))
    return normalized_value

def normalize_target(value):
    normalized_value = (value - (-65)) / (-8 - (-65))
    return normalized_value

def normalize_mt(value):
    normalized_value = (value - 5) / (95 - 5)
    return normalized_value

def GCPercent(seq):                       
    counts = collections.Counter(seq)
    percent = (counts['C'] + counts['G'])/float(len(seq)) 
    return percent

file_paths = [
    'Feature_MLP_conventional.txt'
]

for file_path in file_paths:
    with open(file_path, 'w') as file:
        pass  

# Define NUPACK model-same as CRISPR trans-cleavage reaction
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

# Load crRNA sequences
file1 = open('merged_Cas13a_1e11_full_guide_sequences_unique_before_19min.txt', 'r')    
lines = file1.readlines()
guide_array = [line.strip() for line in lines]

# Load target sequences
file2 = open('merged_Cas13a_1e11_target_sequences_noPFS_unique_before_19min.txt', 'r')    
lines = file2.readlines()
full_target_array = [line.strip() for line in lines]
file3 = open('merged_Cas13a_1e11_target_sequences_noPFS_unique_before_19min.txt', 'r')    
lines = file3.readlines()
truncated_target_array = [line.strip() for line in lines]
file4 = open('merged_Cas13a_1e11_full_target_sequences_unique_before_19min.txt', 'r')   
lines = file4.readlines()
full_target_array_complete = [DNA_to_RNA(line.strip()) for line in lines]

min_guide_energy = np.inf
max_guide_energy = -np.inf
min_target_bh_energy = np.inf
max_target_bh_energy = -np.inf
min_target_energy = np.inf
max_target_energy = -np.inf

 # Initialize energy_array
energy_array = []

for i in range (0, len(guide_array)):
    energy_array_unit = []

    guide = guide_array[i]
    guide_spacer = guide[-28:]
    target_full = full_target_array[i]
    target_full_complete = full_target_array_complete[i]
    target_truncated = truncated_target_array[i]

    start_positions = find_substring_positions(target_truncated, target_full_complete)
    start_position = start_positions[0]

    # Initialize energy_array
    energy_array_guide = []
    energy_array_target = []

    # Compute ensemble energy
    partition_function_guide = pfunc(strands=guide, model=my_model_RNA)
    ensemble_energy_guide = partition_function_guide[1]
    partition_function_target_bh = pfunc(strands=target_full_complete, model=my_model_RNA)
    ensemble_energy_target_bh = partition_function_target_bh[1]
    partition_function_target = pfunc(strands=[guide, target_truncated], model=my_model_RNA)
    ensemble_energy_target = partition_function_target[1]
    
    # Compute suboptimal structures and energy
    subopt_structures_guide = subopt(strands=guide, energy_gap=3, model=my_model_RNA)  
    subopt_structures_target_bh = subopt(strands=target_full_complete, energy_gap=0.1, model=my_model_RNA)
    subopt_structures_target = subopt(strands=[guide, target_truncated], energy_gap=3, model=my_model_RNA)

    # Get paired base positions in ssRNA target before hybridization
    left_paren_target_bh, right_paren_target_bh = find_pair(str(subopt_structures_target_bh[0].structure))
    corrected_target_structure_bh = pairing_state_correction(str(subopt_structures_target_bh[0].structure), left_paren_target_bh, right_paren_target_bh, start_position, start_position+27)

    len_scaffold = 36
    len_spacer = 28

    # Calculate melting temperature for guide and target in double-stranded form (using Biopython)
    Tm_guide = mt.Tm_NN(guide_spacer, Na=100, Mg=10, nn_table=mt.RNA_NN1)
    Tm_target = mt.Tm_NN(target_truncated, Na=100, Mg=10, nn_table=mt.RNA_NN1)

    # Calculate GC content
    GC_guide = GCPercent(guide_spacer)
    GC_target = GCPercent(target_truncated)
    
    # Calculate/compare subopt_structures_guide[0].energy and subopt_structures_target[0].energy
    if subopt_structures_guide[0].energy + 5.08 > max_guide_energy:
        max_guide_energy = subopt_structures_guide[0].energy + 5.08
    if subopt_structures_guide[0].energy + 5.08 < min_guide_energy:
        min_guide_energy = subopt_structures_guide[0].energy + 5.08

    standard_energy_target_bh = structure_energy(strands=target_full_complete, structure=corrected_target_structure_bh, model=my_model_RNA)
    if subopt_structures_target_bh[0].energy - standard_energy_target_bh > max_target_bh_energy:
        max_target_bh_energy = subopt_structures_target_bh[0].energy - standard_energy_target_bh
    if subopt_structures_target_bh[0].energy - standard_energy_target_bh < min_target_bh_energy:
        min_target_bh_energy = subopt_structures_target_bh[0].energy - standard_energy_target_bh
    
    if subopt_structures_target[0].energy > max_target_energy:
        max_target_energy = subopt_structures_target[0].energy
    if subopt_structures_target[0].energy < min_target_energy:
        min_target_energy = subopt_structures_target[0].energy

    to_be_added = [normalize_mt(Tm_guide)]
    to_be_added.append(normalize_mt(Tm_target))

    to_be_added.append(GC_guide)
    to_be_added.append(GC_target)

    to_be_added.append(normalize_guide(subopt_structures_guide[0].energy + 5.08))
    to_be_added.append(normalize_target_bh(subopt_structures_target_bh[0].energy - standard_energy_target_bh))
    to_be_added.append(normalize_target(subopt_structures_target[0].energy))

    energy_array_unit.append(to_be_added)

    energy_array_to_append = [energy_array_unit]
    energy_array.append(energy_array_to_append)

energy_array = list(itertools.chain.from_iterable(energy_array))

# Open a file in write mode
with open('Feature_MLP_conventional.txt', 'a') as file:
    # Iterate over each row in the 2D array
    for row in energy_array:
        # Convert each element to a string and join them with spaces
        file.write(' '.join(map(str, row)) + '\n')
        

print('-------------')

print(min_guide_energy, max_guide_energy, min_target_bh_energy, max_target_bh_energy, min_target_energy, max_target_energy)